In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import datetime
import random
import uvicorn

app = FastAPI(title="Weather Forecast Chatbot API")

# =========================
# MOCK HÀM DỰ BÁO
# Sau này bạn thay bằng model thật của bạn
# =========================
def get_weather_prediction():
    """
    Hàm này hiện đang giả lập output.
    Sau này bạn thay bằng:
    - rf_model.predict(...)
    - xgb_model.predict(...)
    - lgb_model.predict(...)
    - pitan / lstm predict(...)
    """
    today = datetime.datetime.now()
    tomorrow = today + datetime.timedelta(days=1)

    possible_status = [
        "Clear sky",
        "Partly cloudy",
        "Overcast",
        "Moderate rain",
        "Light drizzle"
    ]

    final_status = random.choice(possible_status)
    final_temp = round(random.uniform(27, 34), 1)
    final_humidity = round(random.uniform(65, 95), 1)

    if final_humidity > 80:
        advice = "Độ ẩm cao, có khả năng xuất hiện mưa rào. Bạn nên mang theo ô hoặc áo mưa."
    elif final_temp > 33:
        advice = "Ngày mai trời khá nóng. Bạn nên uống đủ nước và hạn chế ra ngoài buổi trưa."
    else:
        advice = "Thời tiết khá ổn định, phù hợp cho hoạt động ngoài trời."

    return {
        "date": tomorrow.strftime("%d/%m/%Y"),
        "status": final_status,
        "temperature": final_temp,
        "humidity": final_humidity,
        "advice": advice
    }

# =========================
# REQUEST MODEL
# =========================
class ChatRequest(BaseModel):
    message: str

# =========================
# ROUTES
# =========================
@app.get("/")
def home():
    return {"message": "Weather Forecast Chatbot API is running."}

@app.get("/predict")
def predict_weather():
    result = get_weather_prediction()
    return result

@app.post("/chat")
def chat_weather(req: ChatRequest):
    user_message = req.message.lower()
    forecast = get_weather_prediction()

    if "mai" in user_message or "ngày mai" in user_message or "thời tiết" in user_message:
        reply = (
            f"Dự báo ngày {forecast['date']}:\n"
            f"- Trạng thái: {forecast['status']}\n"
            f"- Nhiệt độ trung bình: {forecast['temperature']}°C\n"
            f"- Độ ẩm: {forecast['humidity']}%\n"
            f"- Khuyến nghị: {forecast['advice']}"
        )
    elif "nhiệt độ" in user_message:
        reply = f"Nhiệt độ dự kiến ngày {forecast['date']} là khoảng {forecast['temperature']}°C."
    elif "độ ẩm" in user_message:
        reply = f"Độ ẩm dự kiến ngày {forecast['date']} là khoảng {forecast['humidity']}%."
    elif "mưa" in user_message:
        if "rain" in forecast["status"].lower() or forecast["humidity"] > 80:
            reply = f"Khả năng có mưa tương đối cao vào ngày {forecast['date']}. {forecast['advice']}"
        else:
            reply = f"Khả năng mưa không cao vào ngày {forecast['date']}."
    else:
        reply = (
            "Xin chào, tôi là chatbot dự báo thời tiết TP.HCM.\n"
            "Bạn có thể hỏi:\n"
            "- Thời tiết ngày mai thế nào?\n"
            "- Nhiệt độ ngày mai bao nhiêu?\n"
            "- Có mưa không?\n"
            "- Độ ẩm ngày mai thế nào?"
        )

    return {"reply": reply}

# Chạy API server
if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)